In [9]:
import os
import numpy as np
import pandas as pd

In [10]:
data_dir = 'C:\\Users\\Ben\\Box\\duke_bilbo\\MemberFolders\\Ben\\statistical_learning\\datascience_projects\\cms_data\\Medicare Current Beneficiary Survey - Survey File_OLD\\Medicare Current Beneficiary Survey - Survey File'

In [11]:
csvs = [i for i in os.listdir(os.path.join(data_dir,'all_years_combined')) if '.csv' in i]
fall_csvs = [i for i in csvs if 'fall' in i]

In [13]:
all_fall_surveys_combined = pd.DataFrame()

for table in fall_csvs:
    df = pd.read_csv(os.path.join(data_dir, 'all_years_combined', table))
    cols_2_drop = [i for i in df.columns if 'PUFF' in i]
    int_df = df.drop(columns = cols_2_drop)
    int_df['combined_ID'] = int_df['PUF_ID'].astype('str') +  '_' + int_df['SURVEYYR'][1].astype('str')
    all_fall_surveys_combined = pd.concat([all_fall_surveys_combined, int_df])
    
all_fall_surveys_combined = all_fall_surveys_combined.set_index('combined_ID')

C:\Users\Ben\AppData\Local\Temp\ipykernel_33320\932248223.py:4: DtypeWarning: Columns (44,47,50,51,53,54,75,80,82,83,87,91,92,93,95,97,101,104,110,112,117,118,119,120,126,128,139,140,141,142,143,144,172,175,186,187,192,193,194,195,196,198,199,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,225,227,229,231,235) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(data_dir, 'all_years_combined', table))
C:\Users\Ben\AppData\Local\Temp\ipykernel_33320\932248223.py:4: DtypeWarning: Columns (43,51,53,70,77,78,80,84,86,97,103,104,105,107,110,111,112,113,116,119,121,125,132,133,134,135,136,137,178,179,181,186,187,188,189,190,192,193,194,203,204,205,206,207,208,209,210,212,213,214,215,216,217,219,221,223,225) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(data_dir, 'all_years_combined', table))
C:\Users\Ben\AppData\Local\Temp\ipykernel_33320\932248223.py:4: DtypeWarning: 

In [21]:
all_fall_surveys_combined### this is to add the raw PUF ids to the combined dataframe
### this will help drastically with leakage as we are not incl. the same
### beneficiary multiple times when training the model

puf_ids_raw = []
for row in range(len(all_fall_surveys_combined)):
    puf_ids_raw = np.append(puf_ids_raw, int(str(all_fall_surveys_combined['PUF_ID'][row])[2:]))
all_fall_surveys_combined['PUF_ID_NOY'] = puf_ids_raw

C:\Users\Ben\AppData\Local\Temp\ipykernel_33320\1687958254.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  puf_ids_raw = np.append(puf_ids_raw, int(str(all_fall_surveys_combined['PUF_ID'][row])[2:]))


In [23]:
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(test_size=0.2, random_state=4)
train_idx, test_idx = next(gss.split(all_fall_surveys_combined, groups=all_fall_surveys_combined['PUF_ID_NOY']))
data = (all_fall_surveys_combined.replace({'D': np.nan, 'R': np.nan}).apply(pd.to_numeric, errors='coerce'))

In [24]:
### Missing survey responses were handled using median imputation with missingness indicators for continuous variables, 
### this approach preserves information while minimizing bias in predictive modeling.”

from sklearn.impute import SimpleImputer
num_cols = data.select_dtypes(include=['int64', 'float64']).columns
imputer = SimpleImputer(strategy='median')
data[num_cols] = imputer.fit_transform(data[num_cols])

for col in num_cols:
    data[f'{col}_missing'] = data[col].isna().astype(int)

C:\Users\Ben\AppData\Local\Temp\ipykernel_33320\697087684.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[f'{col}_missing'] = data[col].isna().astype(int)
C:\Users\Ben\AppData\Local\Temp\ipykernel_33320\697087684.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data[f'{col}_missing'] = data[col].isna().astype(int)
C:\Users\Ben\AppData\Local\Temp\ipykernel_33320\697087684.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perfor

In [25]:
data

,PUF_ID,SURVEYYR,VERSION,ADM_H_MEDSTA,ADM_H_GHPSW,ADM_H_PDRS,ADM_OP_MDCD,ADM_DUAL_FLAG_YR,ADM_FFS_FLAG_YR,ADM_MA_FLAG_YR,...,MA_MADVRX_missing,MA_MADVDOC_missing,MA_MADVLAB_missing,MA_MADVINPT_missing,MA_MADVHEAR_missing,MA_MADVBEH_missing,HLT_TEETHGUM_missing,HLT_DRYMOUTH_missing,HLT_TOOTHSEN_missing,PUF_ID_NOY_missing
combined_ID,,,,,,,,,,,,,,,,,,,,,
18000001_2018,18000001.0,2018.0,1.0,1.0,1.0,1.0,2.0,3.0,1.0,3.0,...,0,0,0,0,0,0,0,0,0,0
18000004_2018,18000004.0,2018.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0,...,0,0,0,0,0,0,0,0,0,0
18000005_2018,18000005.0,2018.0,1.0,2.0,2.0,1.0,1.0,1.0,3.0,1.0,...,0,0,0,0,0,0,0,0,0,0
18000006_2018,18000006.0,2018.0,1.0,1.0,2.0,1.0,1.0,1.0,3.0,1.0,...,0,0,0,0,0,0,0,0,0,0
18000007_2018,18000007.0,2018.0,1.0,1.0,2.0,1.0,1.0,1.0,3.0,1.0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17029985_2017,17029985.0,2017.0,1.0,1.0,2.0,1.0,1.0,1.0,3.0,1.0,...,0,0,0,0,0,0,0,0,0,0
17029988_2017,17029988.0,2017.0,1.0,1.0,2.0,1.0,1.0,1.0,3.0,1.0,...,0,0,0,0,0,0,0,0,0,0
17029993_2017,17029993.0,2017.0,1.0,1.0,2.0,3.0,1.0,1.0,3.0,1.0,...,0,0,0,0,0,0,0,0,0,0


In [26]:
data.to_parquet('all_fall_surveys_combined.parquet')

In [27]:
os.getcwd()

'C:\\Users\\Ben\\Box\\duke_bilbo\\MemberFolders\\Ben\\statistical_learning\\datascience_projects\\cms_data\\Medicare Current Beneficiary Survey - Survey File_OLD\\Medicare Current Beneficiary Survey - Survey File'

## creating metadata parquet

In [28]:
data_direc = 'C:\\Users\\Ben\\Box\\duke_bilbo\\MemberFolders\\Ben\\statistical_learning\\datascience_projects\\cms_data\\Medicare Current Beneficiary Survey - Survey File\\2023\\SFPUF2023_Data'
### importing and cleaning the metadata
metadata = pd.read_table(os.path.join(data_direc, 'column_labels.txt'), header = None)
acronym = []
full_name = []
for row in range(len(metadata)):
    acronym = np.append(acronym, metadata[0][row].split('"')[0].split('=')[0].split(' ')[0])
    full_name = np.append(full_name, metadata[0][row].split('=')[1].split('"')[1])
metadata['acronym'] = acronym
metadata['full_name'] = full_name
metadata = metadata.drop(columns = {0})
metadata = metadata.set_index('acronym')

In [30]:
metadata.to_parquet('full_feature_names.parquet')